# Tutorial 3 — Physics Applications

Lévy processes appear throughout statistical physics:
- **Anomalous diffusion** — fractional BM and α-stable processes
- **First passage times** — exact Lévy distribution
- **Stochastic volatility via subordination** — random time changes
- **Hawkes processes** — self-exciting point processes in neuroscience and seismology

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spxa.zoo.levy import BrownianMotion, AlphaStable
from spxa.zoo.beyond import FractionalBrownianMotion, HawkesProcess, OULevy
from spxa.zoo.levy import GammaProcess
from spxa.sim import first_passage_time_exact_bm, first_passage_time_bm

## 1. Anomalous diffusion: fractional Brownian motion

Standard BM has $\text{Var}(B_t) = \sigma^2 t$ (linear in $t$). Fractional BM with Hurst index $H$ has
$$\text{Var}(B^H_t) = \sigma^2 t^{2H}$$
- $H = 0.5$: standard diffusion (Brownian)
- $H > 0.5$: superdiffusion (long memory, persistent increments)
- $H < 0.5$: subdiffusion (anti-persistent increments)

In [ ]:
rng = np.random.default_rng(42)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, H, label in zip(axes, [0.3, 0.5, 0.8], ['H=0.3 (subdiffusion)', 'H=0.5 (BM)', 'H=0.8 (superdiffusion)']):
    fbm = FractionalBrownianMotion(H=H)
    paths = fbm.simulate(n_steps=1000, n_paths=5, T=1.0, rng=rng)
    t = np.linspace(0, 1, 1001)
    for i in range(5):
        ax.plot(t, paths[i], lw=0.7)
    ax.set_title(label)
    ax.set_xlabel('Time')
plt.suptitle('Fractional Brownian motion paths')
plt.tight_layout()
plt.savefig('fbm_paths.png', dpi=120)
plt.show()

## 2. Mean-square displacement

MSD$(t) = \mathbb{E}[|X_t - X_0|^2]$ is a key observable in single-particle tracking experiments.
For fBM: MSD$(t) = 2\sigma^2 t^{2H}$.

In [ ]:
rng2 = np.random.default_rng(1)
t_vals = np.array([0.01, 0.05, 0.1, 0.2, 0.5, 1.0])
fig, ax = plt.subplots(figsize=(8, 5))
for H, color in zip([0.3, 0.5, 0.7], ['blue', 'green', 'red']):
    fbm = FractionalBrownianMotion(H=H, sigma=1.0)
    msds = []
    for t in t_vals:
        paths = fbm.simulate(n_steps=100, n_paths=1000, T=t, rng=rng2)
        msds.append(np.mean(paths[:, -1]**2))
    ax.loglog(t_vals, msds, 'o-', color=color, label=f'H={H}')
    theory = 2.0 * t_vals**(2 * H)
    ax.loglog(t_vals, theory, '--', color=color, alpha=0.5)

ax.set_xlabel('Time $t$')
ax.set_ylabel('MSD')
ax.set_title('Mean-square displacement: empirical (solid) vs $2t^{2H}$ (dashed)')
ax.legend()
plt.tight_layout()
plt.savefig('msd.png', dpi=120)
plt.show()

## 3. Heavy-tailed diffusion: α-stable processes

For $\alpha < 2$, the $\alpha$-stable process has power-law tails: no finite variance, no finite mean for $\alpha \leq 1$.
This describes Lévy flights observed in animal foraging, plasma turbulence, and financial returns.

In [ ]:
rng3 = np.random.default_rng(2)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, alpha, name in zip(axes, [2.0, 1.5, 1.0], ['α=2 (Gaussian)', 'α=1.5 (stable)', 'α=1 (Cauchy)']):
    proc = AlphaStable(alpha=alpha, beta=0.0, sigma=1.0)
    paths = proc.simulate(n_steps=200, n_paths=5, T=1.0, rng=rng3)
    t = np.linspace(0, 1, 201)
    for i in range(5):
        ax.plot(t, paths[i], lw=0.7)
    ax.set_title(name)
    ax.set_xlabel('Time')
plt.suptitle('α-stable sample paths (decreasing α → heavier tails, larger jumps)')
plt.tight_layout()
plt.savefig('stable_paths.png', dpi=120)
plt.show()

## 4. First passage times

The time for BM to hit a level $a > 0$ follows the Lévy distribution:
$$P(\tau \leq t) = 2\Phi(-a/(\sigma\sqrt{t}))$$
This is an $\alpha=1/2$ stable distribution — heavy-tailed with infinite mean.

In [ ]:
rng4 = np.random.default_rng(3)
tau_exact = first_passage_time_exact_bm(level=1.0, sigma=1.0, n_samples=20000, rng=rng4)
tau_sim   = first_passage_time_bm(level=1.0, sigma=1.0, T_max=20.0, n_steps=5000, n_paths=3000, rng=rng4)
tau_sim   = tau_sim[np.isfinite(tau_sim)]

fig, ax = plt.subplots(figsize=(8, 5))
from scipy.stats import levy
t_range = np.linspace(0.01, 8, 500)
ax.hist(tau_exact, bins=80, density=True, alpha=0.5, label='Exact samples', color='steelblue')
ax.hist(tau_sim[tau_sim < 8], bins=80, density=True, alpha=0.4, label='Simulation', color='orange')
c = 1.0  # level=1, sigma=1 → Lévy(0, c=1)
ax.plot(t_range, levy.pdf(t_range, scale=c), 'k-', lw=2, label='Lévy pdf')
ax.set_xlim(0, 8); ax.set_xlabel('First passage time τ'); ax.set_ylabel('Density')
ax.set_title('First passage time of BM to level 1')
ax.legend()
plt.tight_layout()
plt.savefig('first_passage.png', dpi=120)
plt.show()
print(f'Empirical median: {np.median(tau_exact):.4f}  (theory: 1.0)')

## 5. Hawkes process — self-exciting events

The Hawkes process models event clustering: each event increases the probability of future events.
Applications: earthquake aftershocks, neural spike trains, high-frequency order book arrivals.

In [ ]:
rng5 = np.random.default_rng(4)
hawkes = HawkesProcess(mu=1.0, alpha=0.6, beta=2.0)
print(f'Mean intensity λ̄ = {hawkes.mean_intensity:.4f}')
print(f'Fano factor F    = {hawkes.fano_factor:.4f}  (>1: overdispersed)')

paths_hk = hawkes.simulate(n_steps=1000, n_paths=3, T=20.0, rng=rng5)
t_hk = np.linspace(0, 20, 1001)
fig, ax = plt.subplots(figsize=(11, 4))
for i in range(3):
    ax.plot(t_hk, paths_hk[i], lw=0.8, label=f'Path {i+1}')
ax.set_xlabel('Time'); ax.set_ylabel('N(t)'); ax.set_title('Hawkes process counting paths')
ax.legend()
plt.tight_layout()
plt.savefig('hawkes.png', dpi=120)
plt.show()

## 6. OU-Lévy — stochastic volatility

The Ornstein–Uhlenbeck process driven by a Gamma subordinator is the Barndorff-Nielsen–Shephard stochastic volatility model. The variance process $V_t$ is stationary, non-negative, and mean-reverting.

In [ ]:
rng6 = np.random.default_rng(5)
lam = 2.0
g_driver = GammaProcess(a=1.0, b=2.0)
ou = OULevy(lam=lam, subordinator=g_driver, v0=0.5)

print(f'Stationary mean    : {ou.stationary_mean:.4f}')
print(f'Stationary variance : {ou.stationary_variance:.4f}')
print(f'Autocov at h=1     : {ou.autocovariance(1.0):.6f}  (theory: e^{{-{lam}}}·Var = {np.exp(-lam)*ou.stationary_variance:.6f})')

paths_ou = ou.simulate(n_steps=500, n_paths=3, T=5.0, rng=rng6)
t_ou = np.linspace(0, 5, 501)
fig, ax = plt.subplots(figsize=(10, 4))
for i in range(3):
    ax.plot(t_ou, paths_ou[i], lw=0.8)
ax.axhline(ou.stationary_mean, color='k', ls='--', lw=1, label='Stationary mean')
ax.set_xlabel('Time'); ax.set_ylabel('V(t)'); ax.set_title('OU-Lévy (Barndorff-Nielsen–Shephard variance process)')
ax.legend()
plt.tight_layout()
plt.savefig('ou_levy.png', dpi=120)
plt.show()